# Evidência multimodal — fixtures (Limen Épico 8 / E8.2)

> Protótipo acadêmico — **não** é um dispositivo médico.

Inspeciona fixtures versionadas de **vídeo**, **áudio** e **prescriptions**
e cita o catálogo de datasets/fontes usados só como referência (sem download
no CI).

Seed Compose da demo: `./scripts/seed-multimodal-demo.sh`.

## Catálogo (referência)

| Modalidade | Fonte | URL |
| ---------- | ----- | --- |
| Áudio (ref.) | AudioSet | https://research.google.com/audioset/ |
| Áudio (demo) | Medical Speech / gravação ≤60s | Kaggle medical-speech (ver README fixtures) |
| Vídeo fisio | 3DYoga90 | https://github.com/seonokkim/3dyoga90 |
| Vídeo cirurgia leve | Stock CC | Creative Commons / bancos CC0 |
| Prescrições | Sintético Limen | `data/fixtures/prescriptions/` |

Brutos grandes: `data/raw/` (gitignored).

In [ ]:
from pathlib import Path
import json
import csv

ROOT = Path("..").resolve()
FIX = ROOT / "data" / "fixtures"

video_dir = FIX / "video"
audio_dir = FIX / "audio"
rx_dir = FIX / "prescriptions"

video_files = sorted(video_dir.glob("video_*.avi"))
audio_files = sorted(audio_dir.glob("audio_*.wav"))
rx_files = sorted(rx_dir.glob("prescriptions_*.csv"))

print("video:", [p.name for p in video_files])
print("audio:", [p.name for p in audio_files])
print("prescriptions:", [p.name for p in rx_files])

assert video_files and audio_files and rx_files, "Fixtures multimodais ausentes"

for manifest_path in (video_dir / "manifest.json", audio_dir / "manifest.json", rx_dir / "manifest.json"):
    if manifest_path.is_file():
        meta = json.loads(manifest_path.read_text(encoding="utf-8"))
        print(f"\n{manifest_path.parent.name}/manifest.json keys:", sorted(meta)[:12])

for path in rx_files:
    with path.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    anomalies = sum(1 for r in rows if r.get("label") == "anomaly")
    print(f"{path.name}: n={len(rows)} anomalies={anomalies}")

for path in video_files + audio_files:
    print(f"{path.name}: size={path.stat().st_size} bytes")

## Notas

- Fixtures são **sintéticas** (AVI/WAV/CSV determinísticos) — AudioSet / 3DYoga90
  / Medical Speech / stock CC aparecem no **relatório** e READMEs, não como
  dependência de runtime.
- Regenerar: `scripts/prepare_video_fixtures.py`,
  `scripts/prepare_audio_fixtures.py`,
  `scripts/prepare_prescription_fixtures.py`.
- Sem PHI e sem afirmação diagnóstica.

READMEs:
[`video`](../data/fixtures/video/README.md) ·
[`audio`](../data/fixtures/audio/README.md) ·
[`prescriptions`](../data/fixtures/prescriptions/README.md).